
# Домашнее задание 2

В этом задании три независимые задачи:

1. Ridge регрессия: решение через SVD и оценка качества по Leave-One-Out (формула Вудбери) — **10 баллов**;
2. Обновление QR-разложения через вращения Гивенса — **10 баллов**;
3. Спектральная нормализация для линейного слоя нейросети — **10 баллов**.

**Важно:** За каждую задачу можно получить 10 баллов, но максимальная сумма составляет 20 баллов. То есть можно выбрать и выполнить любые две задачи из трёх.

Во всех задачах требуется самостоятельно вывести нужные формулы и реализовать указанные алгоритмы.


# Задача 1. Ridge регрессия: SVD и Leave-One-Out

## Постановка задачи

В Ridge регрессии нужно найти веса $w$, минимизирующие

$$\min_w \|Xw - y\|^2 + \lambda \|w\|^2,$$

где $X \in \mathbb{R}^{n \times d}$ — матрица признаков, $y \in \mathbb{R}^n$ — целевые значения, $\lambda > 0$ — коэффициент регуляризации.

Требуется:
1. Реализовать решение Ridge регрессии через SVD-разложение матрицы $X$;
2. Реализовать вычисление leave-one-out (LOO) кросс-валидации, используя формулу Вудбери.


### Решение Ridge регрессии через SVD

Пусть $X = U \Sigma V^\top$ — сингулярное разложение, где:
- $U \in \mathbb{R}^{n \times r}$ — левые сингулярные векторы
- $\Sigma \in \mathbb{R}^{r \times r}$ — диагональная матрица сингулярных значений
- $V \in \mathbb{R}^{d \times r}$ — правые сингулярные векторы
- $r = \min(n, d)$ — ранг матрицы

Тогда решение Ridge регрессии:

$$w = V(\Sigma^2 + \lambda I)^{-1} \Sigma U^\top y$$

### Обратная матрица через SVD

Для $A = X^\top X + \lambda I$ имеем:

$$A = X^\top X + \lambda I = V(\Sigma^2 + \lambda I) V^\top,$$

поэтому

$$A^{-1} = V(\Sigma^2 + \lambda I)^{-1} V^\top.$$

### Leave-One-Out через формулу Вудбери

При удалении $i$-го примера $(x_i, y_i)$:

$$X_{-i}^\top X_{-i} = X^\top X - x_i x_i^\top,$$

то есть rank-1 обновление матрицы $A = X^\top X + \lambda I$:

$$A_{-i} = A - x_i x_i^\top.$$

Формула Вудбери для $(A - uv^\top)^{-1}$:

$$(A - uv^\top)^{-1} = A^{-1} + \frac{A^{-1}uv^\top A^{-1}}{1 - v^\top A^{-1}u}.$$

В нашем случае $u = v = x_i$:

$$A_{-i}^{-1} = A^{-1} + \frac{A^{-1}x_i x_i^\top A^{-1}}{1 - x_i^\top A^{-1}x_i}.$$

Далее веса без $i$-го примера:

$$w_{-i} = A_{-i}^{-1} (X^\top y - y_i x_i),$$

а LOO-предсказание:

$$\hat{y}_i^{LOO} = x_i^\top w_{-i}.$$


Реализуйте три функции:

1. `ridge_regression_svd(X, y, lambda_reg)` — решает Ridge регрессию через SVD-разложение матрицы $X$;
2. `compute_A_inv_svd(U, Sigma, Vt, lambda_reg)` — вычисляет обратную матрицу $A^{-1}$, где $A = X^\top X + \lambda I$, используя SVD-разложение $X = U \Sigma V^\top$;
3. `leave_one_out_linear_regression(X, y, lambda_reg)` — по матрице признаков $X$, вектору ответов $y$ и параметру $\lambda$ возвращает LOO-предсказания модели Ridge, используя формулу Вудбери.

При реализации избегайте явного обращения матриц (`np.linalg.inv`), используйте только операции с разложениями и системы линейных уравнений.


In [ ]:
import numpy as np


def ridge_regression_svd(X, y, lambda_reg):
    """Solve Ridge regression using SVD decomposition of X.

    Returns:
    --------
    ndarray, shape (d,)
        Weights vector w that minimizes ||Xw - y||^2 + lambda * ||w||^2
    """
    raise NotImplementedError


def compute_A_inv_svd(U, Sigma, Vt, lambda_reg):
    """Compute inverse matrix A^(-1) using SVD decomposition.

    Given SVD decomposition X = U @ diag(Sigma) @ Vt, computes A^(-1)
    where A = X^T @ X + lambda_reg * I = V @ (Sigma^2 + lambda_reg * I) @ V^T.

    Parameters:
    -----------
    U : ndarray, shape (n, r)
        Left singular vectors of X
    Sigma : ndarray, shape (r,)
        Singular values of X
    Vt : ndarray, shape (r, d)
        Right singular vectors of X (V^T)
    lambda_reg : float
        Regularization parameter

    Returns:
    --------
    ndarray, shape (d, d)
        Inverse matrix A^(-1) = V @ (Sigma^2 + lambda_reg * I)^(-1) @ V^T
    """
    raise NotImplementedError


def leave_one_out_linear_regression(X, y, lambda_reg):
    """Compute LOO predictions for Ridge regression using Woodbury updates.

    Returns:
    --------
    ndarray, shape (n,)
        Leave-one-out predictions for each sample
    """
    raise NotImplementedError


Проверьте корректность и скорость работы реализации на синтетическом датасете:

Сравните скорость работы `leave_one_out_linear_regression` (формула Вудбери) с наивной реализацией (обучение модели $n$ раз, каждый раз исключая один объект и пересчитывая Ridge регрессию).

Используйте синтетический датасет с 300 объектами и 100 признаками.

In [127]:
def generate_synthetic_dataset(n_samples=300, n_features=100, noise_std=0.1, seed=42):
    """Generate synthetic dataset for testing Ridge regression.

    Parameters:
    -----------
    n_samples : int
        Number of samples
    n_features : int
        Number of features
    noise_std : float
        Standard deviation of noise added to target
    seed : int
        Random seed for reproducibility

    Returns:
    --------
    tuple (X, y)
        X: ndarray, shape (n_samples, n_features), feature matrix
        y: ndarray, shape (n_samples,), target vector
    """
    np.random.seed(seed)

    # Generate random data
    X = np.random.randn(n_samples, n_features)
    # Create target as linear combination with some noise
    true_weights = np.random.randn(n_features)
    y = X @ true_weights + noise_std * np.random.randn(n_samples)

    return X, y


# Example usage:
# X_test, y_test = generate_synthetic_dataset(n_samples=300, n_features=100)
# lambda_reg = 0.1

---

# Задача 2. Обновление QR-разложения через вращения Гивенса

## Постановка задачи

Пусть для матрицы $A \in \mathbb{R}^{m \times n}$ известно QR-разложение $A = QR$, где $Q \in \mathbb{R}^{m \times m}$ — ортогональная матрица, а $R \in \mathbb{R}^{m \times n}$ — верхнетреугольная матрица.

При добавлении новой строки $a^\top \in \mathbb{R}^{1 \times n}$ получаем расширенную матрицу

$$\tilde A = \begin{bmatrix} A \\ a^\top \end{bmatrix} \in \mathbb{R}^{(m+1) \times n}.$$

Требуется реализовать алгоритм обновления QR-разложения для матрицы $\tilde A$ без полного пересчёта, используя вращения Гивенса.

### Алгоритм обновления QR-разложения

Чтобы получить QR-разложение $\tilde A = \tilde Q \tilde R$ без полного пересчёта, можно:

1. Добавить строку $a^\top$ к матрице $R$, получив матрицу $\begin{bmatrix} R \\ a^\top \end{bmatrix}$;
2. Последовательно обнулять элементы под диагональю в новой строке с помощью **вращений Гивенса**, корректируя при этом и матрицу $Q$.

Вращения Гивенса применяются к парам строк, чтобы обнулить элементы под главной диагональю, сохраняя при этом ортогональность матрицы $Q$.

В качестве тестового примера используйте синтетическую матрицу.


Реализуйте две функции:

1. `givens_rotation(a, b)` — вычисляет параметры вращения Гивенса $(c, s)$ для обнуления второго элемента в векторе $[a, b]$;

   Вращение Гивенса обнуляет второй элемент вектора $[a, b]^\top$:

   $$G \begin{bmatrix} a \\ b \end{bmatrix} = \begin{bmatrix} r \\ 0 \end{bmatrix}, \quad \text{где } G = \begin{bmatrix} c & s \\ -s & c \end{bmatrix}, \quad r = \sqrt{a^2 + b^2}.$$

   Параметры $(c, s)$ вычисляются следующим образом:

   - Если $r = 0$: $c = 1$, $s = 0$
   - Иначе: $c = \frac{a}{r}$, $s = \frac{b}{r}$

2. `update_qr_givens(Q, R, new_row)` — по уже известному разложению $A = QR$ строит разложение добавленной матрицы $$\widetilde{A} = \begin{bmatrix} A \\ \text{new\_row} \end{bmatrix}.$$

Проверьте корректность реализации, сравнив результат с полным QR-разложением матрицы $\tilde A$ через `np.linalg.qr`.


In [128]:
def givens_rotation(a, b):
    """Compute Givens rotation parameters (c, s) that zero-out b.

    The rotation is applied to vector [a, b] so that the new second element becomes zero.

    Returns:
    --------
    tuple (c, s)
        Parameters of Givens rotation: c (cosine) and s (sine) as floats
    """
    raise NotImplementedError


def update_qr_givens(Q, R, new_row):
    """Update QR factorization when adding a new row using Givens rotations.

    Given current factorization A = Q @ R, after adding row `new_row` we need
    to restore the upper-triangular structure of R using Givens rotations.

    Returns:
    --------
    tuple (Q_new, R_new)
        Updated orthogonal matrix Q_new and upper-triangular matrix R_new
        such that [A; new_row] = Q_new @ R_new
    """
    raise NotImplementedError


In [ ]:
# Example usage and testing:
# Generate a random matrix A
# A = np.random.randn(10, 5)
# Q, R = np.linalg.qr(A)
#
# # Add a new row
# new_row = np.random.randn(5)
# Q_new, R_new = update_qr_givens(Q, R, new_row)
#
# # Verify correctness
# A_extended = np.vstack([A, new_row])
# Q_full, R_full = np.linalg.qr(A_extended)
#
# # Check that Q_new @ R_new ≈ A_extended
# # and compare with Q_full, R_full


---

# Спектральная нормализация

## Задача 3. Спектральная нормализация

[Спектральная нормализация](https://arxiv.org/abs/1802.05957) — один из популярных видов нормализации в нейросетях, который особенно часто применяется в генеративно-состязательных сетях.
В PyTorch спектральная нормализация реализована в виде функции [spectral_norm](https://pytorch.org/docs/stable/generated/torch.nn.utils.spectral_norm.html).
Эта функция модифицирует нейросеть, и в нужный слой сети добавляется:
- Вычисление оценки спектральной нормы (наибольшего сингулярного числа) матрицы с помощью степенного метода,
- Нормировка параметра линейного слоя с помощью этой оценки

### Задание
1. (3 балла) Аналитически найдите константу Липшица для функции умножения матрицы на вектор, где матрица есть результат применения спектральной нормализации, если старшее сингулярное число вычислено точно.
2. (7 баллов) По умолчанию спектральная нормализация в PyTorch делает одну итерацию степенного метода. Достаточно ли этого?
    1. Возьмите в качестве примера последний слой архитектуры ResNet-50, обученной на ImageNet (`resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)`, из `torchvision`).
    2. Реализуйте алгоритм вычисления старшего сингулярного значения на основе степенного метода.
    3. Сгенерируйте много различных начальных приближений и оцените среднее и разброс результатов степенного метода.
    4. Изобразите среднее и дисперсии для разного количества шагов в виде "ящиков с усами".
    5. Проанализируйте результаты, используя теорию сходимости степенного метода.


In [ ]:
import numpy as np
from torchvision.models import resnet50, ResNet50_Weights

# Example: obtain weight matrix of the last fully connected layer
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
weight_matrix = model.fc.weight.data.numpy()

# Implement power method experiments for spectral normalization below using `weight_matrix`.

def power_method(matrix, num_iterations, initial_vector=None):
    """Compute the largest singular value and corresponding singular vector using power method.

    Returns:
    --------
    tuple (sigma, u)
        sigma: float, the largest singular value
        u: ndarray, shape (min(m, n),), the left singular vector corresponding to sigma
    """
    # TODO
    pass
